# Цаг агаарын заалтын архив

Ус цаг уур, орчны шинжилгээний газрын ажиглалтыг цаг тутам татаж
Feature Service рүү нэмнэ.

**Яагаад хэрэгтэй вэ.** `weather.gov.mn`-ий API нь ЗӨВХӨН хамгийн сүүлийн
заалтыг буцаадаг — өдөр, сар, жилийн цуваа авах зам байхгүй. Тиймээс
хандлага, уур амьсгалын өөрчлөлтийг харах ганц арга бол заалтыг тухай
бүрд нь өөрсдөө хадгалж эхлэх.

**Яагаад Notebook дээр вэ.** Давхарга нь `environment.ub.gov.mn` дээр
бөгөөд тэр хост нь интернетээс хаалттай (GitHub-ийн сервер 443 порт руу
холбогдож чаддаггүй). Notebook нь танай Enterprise дотор ажилладаг тул
давхаргад шууд хүрнэ.

**Хуваарь.** Notebook-ийг хадгалсны дараа item-ийн хуудсан дээрх
*Schedule* товчоор цаг тутам ажиллуулна.

## Тохиргоо

`GIS("home")` нь Notebook доторх идэвхтэй бүртгэлээр автоматаар нэвтэрнэ —
нууц үг, токен бичих шаардлагагүй.

In [ ]:
import json
import urllib.request
from datetime import datetime, timezone

from arcgis.gis import GIS
from arcgis.features import FeatureLayer

# Давхаргын хаяг. Төгсгөлийн /0 нь давхаргын дугаар.
LAYER_URL = (
    "https://environment.ub.gov.mn/hosting/rest/services"
    "/Hosted/Tsag_agaar_arhiv/FeatureServer/0"
)

# "capital" — нийслэлийн 7 станц (жилд ~61 мянган бичлэг)
# "all"     — улсын 317 станц (жилд 2.8 сая — давхаргад хэт хүнд)
SCOPE = "capital"
CAPITAL = "Нийслэл"

API = "https://weather.gov.mn/api/get"

# Хостын урд WAF сууж, танил бус агентыг 403-аар хаадаг.
# Энэ толгойгүй бол хүсэлт чимээгүй унана.
UA = {"User-Agent": "Mozilla/5.0"}

gis = GIS("home")
layer = FeatureLayer(LAYER_URL, gis)
print("Давхарга:", layer.properties.name)

## Эх сурвалжаас татах

In [ ]:
def get(path):
    req = urllib.request.Request(f"{API}/{path}", headers=UA)
    with urllib.request.urlopen(req, timeout=90) as r:
        return json.loads(r.read().decode("utf-8"))


# Станцын бүртгэл нь координат, өндөршлийг өгнө; ажиглалтад тэдгээр байхгүй
stations = {
    s["sid"]: s
    for s in get("obs/aimags")["aimag_sum"]
    if (SCOPE == "all" or s.get("aimag_name") == CAPITAL)
    and s.get("lat") is not None
    and s.get("lon") is not None
}

observations = get("obs/data/aws")["stationAWS"]
print(f"{len(observations)} станцын заалт ирлээ, эдгээрээс {len(stations)}-г хянана")

## Давхардлыг шалгах

⚠ **Энэ алхмыг алгасч БОЛОХГҮЙ.** API нь станц шинэчлэх хүртэл ИЖИЛ
заалтыг буцаасаар байдаг — нийслэлийн хоёр станц (Буянт-Ухаа, Мишээл
Экспо) синоптик цагаар буюу 6 цаг тутам л мэдээлдэг. Шалгахгүй бол цаг
тутам 7 хуулбар үүсч, архив хэдхэн сард ашиглах боломжгүй болно.

Станц бүрийн хамгийн сүүлийн `obs_date`-ыг асууж, түүнээс ХОЙШХИ
заалтыг л нэмнэ.

In [ ]:
latest = {}
result = layer.query(
    where="1=1",
    group_by_fields_for_statistics="sid",
    out_statistics=[{
        "statisticType": "max",
        "onStatisticField": "obs_date",
        "outStatisticFieldName": "last_obs",
    }],
    return_geometry=False,
)
for f in result.features:
    a = f.attributes
    if a.get("sid") is not None and a.get("last_obs") is not None:
        latest[a["sid"]] = a["last_obs"]

print(f"Архивт {len(latest)} станцын бичлэг байна")

## Шинэ заалтыг бэлдэх

In [ ]:
def num(v):
    """Тоо мөн эсэх — эх сурвалж хоосныг null эсвэл мөрөөр өгдөг."""
    return v if isinstance(v, (int, float)) and not isinstance(v, bool) else None


def epoch_ms(iso):
    """ISO мөрийг epoch миллисекунд болгоно. ArcGIS огноог ингэж хадгална."""
    return int(
        datetime.fromisoformat(iso.replace("Z", "+00:00")).timestamp() * 1000
    )


now_ms = int(datetime.now(timezone.utc).timestamp() * 1000)
adds = []

for r in observations:
    st = stations.get(r["sid"])
    if not st:
        continue

    at = epoch_ms(r["obs_date"])

    # Архивт байгаагаас шинэ биш бол алгасна
    seen = latest.get(r["sid"])
    if seen is not None and at <= seen:
        continue

    # pst дээр 0 нь ХЭМЖИГДЭЭГҮЙН тэмдэг — станцын түвшний даралт
    # хэзээ ч тэг болохгүй (сүлжээний бодит доод утга 764 гПа)
    pst = num(r.get("pst"))
    if pst == 0:
        pst = None

    adds.append({
        "geometry": {
            "x": st["lon"],
            "y": st["lat"],
            "spatialReference": {"wkid": 4326},
        },
        "attributes": {
            "sid": r["sid"],
            "name": st.get("sta_name") or st.get("sum_name") or "",
            "place": st.get("sum_name") or "",
            "aimag": st.get("aimag_name") or "",
            "elev": num(st.get("elev")),
            "lat": st["lat"],
            "lon": st["lon"],
            "obs_date": at,
            "logged_at": now_ms,
            "ttt": num(r.get("ttt")),
            "ttt_feels": num(r.get("ttt_feels")),
            "ff": num(r.get("ff")),
            "pst": pst,
            "nh": num(r.get("nh")),
            "wind_speed": num(r.get("wind_speed")),
            "wind_dir": num(r.get("wind_dir")),
            "precip": num(r.get("precip")),
            # Албан жагсаалтад байгаа ч хариултад үргэлж ирдэггүй
            "snow_depth": num(r.get("snow_depth")),
            "tmin": num(r.get("tmin")),
            "tmax": num(r.get("tmax")),
        },
    })

print(f"{len(adds)} шинэ заалт бэлдэгдлээ")

## Архивт нэмэх

In [ ]:
if not adds:
    print("Шинэ заалт алга — бүх станц өмнөх байдлаараа байна")
else:
    res = layer.edit_features(adds=adds)
    ok = [x for x in res.get("addResults", []) if x.get("success")]
    bad = [x for x in res.get("addResults", []) if not x.get("success")]
    print(f"{len(ok)} заалт архивт нэмэгдлээ")
    if bad:
        raise RuntimeError(f"{len(bad)} бичлэг нэмэгдсэнгүй: {bad[0]}")

## Хуваарьт оруулах

1. Notebook-оо **хадгал** (Save)
2. Notebook-ийн **item хуудас** руу оч
3. **Schedule** товч → *Repeat every* **1 hour** → эхлэх огноо сонгоод хадгал

Ажиллагааны түүх нь мөн тэр хуудсан дээр харагдана.

**Шалгах:** эхний удаа `7 заалт архивт нэмэгдлээ`, шууд дахин
ажиллуулбал `Шинэ заалт алга` гэж гарах ёстой — давхардлын хамгаалалт
ажиллаж байна гэсэн үг.